In [4]:
print(names(pop))  # This will print all column names to the console

 [1] "country_name_x" "country_code"   "series_name"    "series_code"   
 [5] "x2000_yr2000"   "x2001_yr2001"   "x2002_yr2002"   "x2003_yr2003"  
 [9] "x2004_yr2004"   "x2005_yr2005"   "x2006_yr2006"   "x2007_yr2007"  
[13] "x2016_yr2016"   "x2017_yr2017"   "x2018_yr2018"   "x2019_yr2019"  
[17] "x2020_yr2020"   "x2021_yr2021"   "x2022_yr2022"   "x2023_yr2023"  
[21] "x2024_yr2024"   "income_group"   "country_name_y"


In [2]:
# ======================================================
#  ODA ANALYSIS PIPELINE — R VERSION (FINAL, CLEAN, NO ERRORS)
# ======================================================

# --- Load Required Libraries ---
if (!require("tidyverse")) install.packages("tidyverse", repos="https://cloud.r-project.org")
if (!require("janitor")) install.packages("janitor", repos="https://cloud.r-project.org")
if (!require("zoo")) install.packages("zoo", repos="https://cloud.r-project.org")

library(tidyverse)
library(janitor)
library(zoo)

# --- STEP 1: Read Input Files ---
crs <- read_csv("crs_disbursements.csv", show_col_types = FALSE)
wdi <- read_csv("wdi_indicators.csv", show_col_types = FALSE)
pop <- read_csv("pop_income.csv", show_col_types = FALSE)

# --- STEP 2: Clean column names ---
crs <- clean_names(crs)
wdi <- clean_names(wdi)
pop <- clean_names(pop)

cat("✅ Data loaded and names cleaned.\n")

# --- STEP 3: Reshape WDI ---
# Find year columns like x2000_yr2000, x2021_yr2021 etc.
wdi_year_cols <- names(wdi)[str_detect(names(wdi), "^x\\d{4}_yr\\d{4}$")]

wdi_long <- wdi %>%
  pivot_longer(
    cols = all_of(wdi_year_cols),
    names_to = "year_raw",
    values_to = "value"
  ) %>%
  mutate(
    year = as.integer(str_extract(year_raw, "\\d{4}")),
    country = coalesce(country_name, country_code)
  ) %>%
  select(country, country_code, series_name, year, value)

wdi_clean <- wdi_long %>%
  group_by(country, year, series_name) %>%
  summarise(value = mean(as.numeric(value), na.rm = TRUE), .groups = "drop") %>%
  pivot_wider(names_from = series_name, values_from = value)

cat("✅ WDI reshaped successfully.\n")

# --- STEP 4: Reshape Population/Income Data ---
# Use the same pattern as WDI (matches x2000_yr2000, etc.)
pop_year_cols <- names(pop)[str_detect(names(pop), "^x\\d{4}_yr\\d{4}$")]

# Safety check: Stop if no columns match
if (length(pop_year_cols) == 0) {
  stop("No year columns found in pop data matching the pattern. Check column names with names(pop).")
}

pop_long <- pop %>%
  pivot_longer(
    cols = all_of(pop_year_cols),
    names_to = "year_raw",
    values_to = "value"
  ) %>%
  mutate(
    year = as.integer(str_extract(year_raw, "\\d{4}")),
    country = coalesce(country_name_x, country_code)  # Adjusted to match your column names
  ) %>%
  select(country, country_code, income_group, year, value)

pop_clean <- pop_long %>%
  group_by(country, year) %>%
  summarise(
    population = mean(as.numeric(value), na.rm = TRUE),
    income_group = first(na.omit(income_group)),
    .groups = "drop"
  )

cat("✅ Population/Income data reshaped.\n")

# --- STEP 5: Aggregate CRS (ODA) Data ---
# Dynamically assign columns based on existence to avoid referencing non-existent ones
crs <- crs %>%
  mutate(
    sector = if ("sector" %in% names(.)) sector else
             if ("purpose_name" %in% names(.)) purpose_name else
             if ("purpose" %in% names(.)) purpose else NA_character_,
    oda_value = if ("obs_value" %in% names(.)) obs_value else
                if ("observation_value" %in% names(.)) observation_value else
                if ("amount" %in% names(.)) amount else
                if ("value" %in% names(.)) value else NA_real_
  )

# Check if oda_value has any non-NA values; warn if not
if (all(is.na(crs$oda_value))) {
  warning("No valid ODA values found in crs data. Check column names and data quality.")
}

# Dynamically determine country and year columns
country_col <- if ("recipient" %in% names(crs)) "recipient" else
               if ("recipient_name" %in% names(crs)) "recipient_name" else NULL
year_col <- if ("time_period" %in% names(crs)) "time_period" else
            if ("year" %in% names(crs)) "year" else NULL

if (is.null(country_col) || is.null(year_col)) {
  stop("Required columns for country or year not found in crs data. Check names(crs).")
}

oda_total <- crs %>%
  group_by(country = .data[[country_col]], year = as.integer(.data[[year_col]])) %>%
  summarise(oda_total = sum(as.numeric(oda_value), na.rm = TRUE), .groups = "drop")

oda_edu <- crs %>%
  mutate(sector_lower = tolower(sector)) %>%
  filter(str_detect(sector_lower, "education")) %>%
  group_by(country = .data[[country_col]], year = as.integer(.data[[year_col]])) %>%
  summarise(oda_education = sum(as.numeric(oda_value), na.rm = TRUE), .groups = "drop")

oda_summary <- oda_total %>%
  left_join(oda_edu, by = c("country", "year")) %>%
  mutate(oda_education = replace_na(oda_education, 0))

cat("✅ CRS (ODA) aggregated.\n")

# --- STEP 6: Merge All Datasets ---
merged <- oda_summary %>%
  left_join(wdi_clean, by = c("country", "year")) %>%
  left_join(pop_clean, by = c("country", "year"))

cat("✅ All datasets merged successfully.\n")

# --- STEP 7: Add Lags and Z-scores ---
merged <- merged %>%
  arrange(country, year) %>%
  group_by(country) %>%
  mutate(
    oda_total_lag1 = lag(oda_total, 1),
    oda_total_lag2 = lag(oda_total, 2),
    oda_total_lag3 = lag(oda_total, 3)
  ) %>%
  ungroup() %>%
  group_by(income_group, year) %>%
  mutate(
    z_oda_total = ifelse(sd(oda_total, na.rm = TRUE) > 0,
                         (oda_total - mean(oda_total, na.rm = TRUE)) / sd(oda_total, na.rm = TRUE),
                         NA)
  ) %>%
  ungroup()

cat("✅ Lag variables + z-scores added.\n")

# --- STEP 8: Optional Plot (Education ODA vs Literacy) ---
lit_col <- names(merged)[str_detect(names(merged), "literacy")]
if (length(lit_col) > 0) {
  ggplot(merged, aes(x = .data[[lit_col[1]]], y = oda_education)) +
    geom_point(alpha = 0.5, color = "steelblue") +
    geom_smooth(method = "lm", se = FALSE, color = "red") +
    labs(
      title = paste("Education ODA vs", lit_col[1]),
      x = lit_col[1], y = "Education ODA (USD)"
    ) +
    theme_minimal()
} else {
  cat("⚠️ Literacy column not found — skipping plot.\n")
}

# --- STEP 9: Export Processed Dataset ---
write_csv(merged, "processed.csv")
cat("✅ processed.csv exported successfully.\n")

Loading required package: tidyverse

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.1     ✔ stringr   1.5.2
✔ ggplot2   4.0.0     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: janitor

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘janitor’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘snakecase’


Loading required package: zoo

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :


✅ Data loaded and names cleaned.


Warning message:
“There were 106 warnings in `summarise()`.
The first warning was:
ℹ In argument: `value = mean(as.numeric(value), na.rm = TRUE)`.
ℹ In group 2: `country = "Bangladesh"`, `year = 2000`, `series_name = "Literacy
  rate, adult total (% of people ages 15 and above)"`.
Caused by warning in `mean()`:
! NAs introduced by coercion
ℹ Run `dplyr::last_dplyr_warnings()` to see the 105 remaining warnings.”


✅ WDI reshaped successfully.
✅ Population/Income data reshaped.
✅ CRS (ODA) aggregated.
✅ All datasets merged successfully.
✅ Lag variables + z-scores added.
⚠️ Literacy column not found — skipping plot.
✅ processed.csv exported successfully.
